In [1]:
# %%
"""
PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10
=======================================================
Runs ONLY the best-performing PTQ configuration identified from the sweep:
static_minmax (FX-graph, MinMax observer, INT8 weights + activations)

Why static_minmax beats dynamic:
Dynamic PTQ quantizes ONLY nn.Linear weights → qint8.
Conv2d layers (which dominate ResNet-50 compute) remain FP32.
Result: negligible size reduction, negligible speedup, same accuracy.

Static PTQ quantizes Conv2d + Linear + residual adds (weights + activations).
Result: ~4× smaller, ~3× faster batch throughput, accuracy drop < 0.0001.

Parameter counting strategy:
    - For nn.Parameter objects: deduplicate by id(p) — safe because each
      Parameter is a distinct Python object.
    - For FX-quantized layers with .weight() callable: deduplicate by
      (module_path, 'weight') string key — avoids data_ptr() aliasing and
      prevents double-counting when a module appears in multiple traversal paths.
    - data_ptr() is intentionally avoided: views/slices of the same storage
      can share data_ptr() causing undercounting.

Mathematical foundation (MinMax observer):
    S = (x_max − x_min) / (2^8 − 1)   [step size; 255 levels for UINT8]
    Z = clip(round(−x_min / S), 0, 255) [zero-point: maps real 0 to grid]
    Q(x) = clamp(round(x / S) + Z, 0, 255)
    x̂   = S · (Q(x) − Z)              [reconstruction]
    |ε|  ≤ S/2                         [max error bounded by half step]

    Per-channel weight quantization: each output filter gets its own (S, Z),
    reducing quantization error vs a single global scale per tensor.

FLOPs note:
    PTQ does not change the computation graph — op count is identical to FP32.
    INT8 batch speedup comes from VNNI/SIMD packing on CPU (onednn backend):
    4 INT8 MACs per instruction vs 1 FP32 FMA, not fewer operations.

Timing note:
    CPU-only script → time.perf_counter() is the correct method.
    CUDA events are not applicable here.
    Warmup runs included to stabilise OS scheduler and cache effects.

Output:
    __1__PTQ_CPU.json
    __1__saved_models_PTQ_CPU/resnet50_static_minmax.pt
"""

"\nPTQ Best Config — static_minmax — ResNet-50 / CIFAR-10\n=======================================================\nRuns ONLY the best-performing PTQ configuration identified from the sweep:\nstatic_minmax (FX-graph, MinMax observer, INT8 weights + activations)\n\nWhy static_minmax beats dynamic:\nDynamic PTQ quantizes ONLY nn.Linear weights → qint8.\nConv2d layers (which dominate ResNet-50 compute) remain FP32.\nResult: negligible size reduction, negligible speedup, same accuracy.\n\nStatic PTQ quantizes Conv2d + Linear + residual adds (weights + activations).\nResult: ~4× smaller, ~3× faster batch throughput, accuracy drop < 0.0001.\n\nParameter counting strategy:\n    - For nn.Parameter objects: deduplicate by id(p) — safe because each\n      Parameter is a distinct Python object.\n    - For FX-quantized layers with .weight() callable: deduplicate by\n      (module_path, 'weight') string key — avoids data_ptr() aliasing and\n      prevents double-counting when a module appears in mu

In [2]:
import os
import json
import time
import copy
import tempfile
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.quantization as tq
from torch.ao.quantization import QConfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score
)

# fvcore preferred for FLOPs; thop as fallback
try:
    from fvcore.nn import FlopCountAnalysis
    _FVCORE = True
except ImportError:
    _FVCORE = False
    from thop import profile as thop_profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON         = "__1__PTQ_CPU.json"
MODEL_SAVE_DIR      = "__1__saved_models_PTQ_CPU"

DEVICE         = torch.device("cpu")
BATCH_SIZE     = 128
NUM_WORKERS    = 2
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 200
WARMUP_RUNS    = 20

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
INPUT_SHAPE = (1, 3, 32, 32)

# ── Best QConfig: MinMax observer ─────────────────────────────────────────────
# Activation : per-tensor affine, MinMax → quint8  (unsigned 0–255)
# Weight     : per-channel symmetric, MinMax → qint8 (signed −128..127)
#              Per-channel: each filter owns its own scale → lower error than
#              a single global scale shared across all output channels.
BEST_QCONFIG = QConfig(
    activation=tq.MinMaxObserver.with_args(
        dtype=torch.quint8,
        qscheme=torch.per_tensor_affine,
    ),
    weight=tq.PerChannelMinMaxObserver.with_args(
        dtype=torch.qint8,
        qscheme=torch.per_channel_symmetric,
    ),
)

In [3]:



# ══════════════════════════════════════════════════════════════════════════════
# Model
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_baseline(path: str) -> nn.Module:
    model = build_model().cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model


# ══════════════════════════════════════════════════════════════════════════════
# Data
# ══════════════════════════════════════════════════════════════════════════════
def get_test_loader() -> DataLoader:
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        root="../data", train=False, download=True, transform=tf
    )
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False,
    )


def get_calib_loader() -> DataLoader:
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = torchvision.datasets.CIFAR10(
        root="../data", train=True, download=True, transform=tf
    )
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(
        sub, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Parameter Counting
# Handles both standard FP32 models and FX-quantized models correctly.
# ══════════════════════════════════════════════════════════════════════════════
def count_params(model: nn.Module, model_label: str = "") -> dict:
    """
    Correct parameter counting for FP32, dynamic PTQ, and FX-quantized models.

    Two deduplication strategies:
    ① nn.Parameter objects  → deduplicate by id(p).
      id() is safe: each Parameter is a distinct Python object even if two
      share underlying storage (views, grouped convolutions).

    ② FX-quantized .weight() callables → deduplicate by (module_name, 'weight').
      After convert_fx(), quantized ops expose weights via .weight() which
      returns a freshly dequantized FloatTensor on each call.
      id() and data_ptr() are both unstable for these — module-path key is safe.

    Why NOT data_ptr():
      Views and slices can share data_ptr() with their base tensor.
      Using data_ptr() as a dedup key can silently merge genuinely separate
      parameters, causing undercounting.
    """
    total, nonzero   = 0, 0
    seen_param_ids   = set()   # for nn.Parameter objects
    seen_module_keys = set()   # for FX .weight() callable modules

    for mod_name, module in model.named_modules():
        # ── ① Standard parameters (FP32, bias, dynamic PTQ) ──────────────────
        for _, p in module.named_parameters(recurse=False):
            pid = id(p)
            if pid in seen_param_ids:
                continue
            seen_param_ids.add(pid)
            n        = p.numel()
            total   += n
            nonzero += int((p != 0).sum().item())

        # ── ② FX quantized weight callable ───────────────────────────────────
        if hasattr(module, "weight") and callable(module.weight):
            key = (mod_name, "weight")
            if key in seen_module_keys:
                continue
            seen_module_keys.add(key)
            try:
                w = module.weight()          # dequantized → FloatTensor
                if isinstance(w, torch.Tensor) and w.numel() > 0:
                    total   += w.numel()
                    nonzero += int((w != 0).sum().item())
            except Exception:
                pass

    if model_label:
        print(
            f"      count_params [{model_label}]: "
            f"total={total:,}  nonzero={nonzero:,}  "
            f"sparsity={1 - nonzero / max(total, 1):.4f}"
        )
    return {"params_total": int(total), "params_nonzero": int(nonzero)}


# ══════════════════════════════════════════════════════════════════════════════
# Disk Size
# Uses the same serialisation format for both models so comparison is valid.
# ══════════════════════════════════════════════════════════════════════════════
def disk_size_mb(model: nn.Module) -> float:
    """
    Serialize via torch.save(model) — full model, not state_dict — so
    that quantized models (which carry their quantization config) are
    measured fairly against the FP32 model serialized the same way.
    Using state_dict for one and full-model for the other would produce
    an unfair comparison.
    """
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs
# fvcore preferred (more accurate for ResNet). thop fallback.
# FLOPs are IDENTICAL for FP32 and INT8 — quantization changes dtype,
# not the number of arithmetic operations.
# ══════════════════════════════════════════════════════════════════════════════
def count_flops(model: nn.Module) -> dict:
    """
    Returns {"flops_G": float, "params_total": int, "params_M": float, "tool": str}.
    Falls back gracefully: fvcore → thop → manual hook-based.
    For FX-quantized models thop cannot hook quantized ops; in that case
    the caller should reuse the FP32 baseline value (FLOPs are identical).
    """
    model = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(*INPUT_SHAPE)

    if _FVCORE:
        try:
            flops = FlopCountAnalysis(model, dummy)
            flops.unsupported_ops_warnings(False)
            total_macs   = flops.total()
            params_total = sum(p.numel() for p in model.parameters())
            return {
                "flops_G"     : round(total_macs * 2 / 1e9, 9),
                "params_total": int(params_total),
                "params_M"    : round(params_total / 1e6, 3),
                "tool"        : "fvcore",
            }
        except Exception:
            pass

    # thop fallback
    try:
        macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        params_total = sum(p.numel() for p in model.parameters())
        return {
            "flops_G"     : round(float(macs) * 2 / 1e9, 9),
            "params_total": int(params_total),
            "params_M"    : round(params_total / 1e6, 3),
            "tool"        : "thop",
        }
    except Exception:
        pass

    # Manual hook-based fallback
    spatial, handles = {}, []

    def make_hook(name):
        def hook(_, __, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))
    with torch.no_grad():
        model(dummy)
    for h in handles:
        h.remove()

    total_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            kh = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            total_flops += 2 * cout * (mod.in_channels // mod.groups) * kh * kw * hout * wout
        elif isinstance(mod, nn.Linear):
            total_flops += 2 * mod.in_features * mod.out_features

    params_total = sum(p.numel() for p in model.parameters())
    return {
        "flops_G"     : round(total_flops / 1e9, 9),
        "params_total": int(params_total),
        "params_M"    : round(params_total / 1e6, 3),
        "tool"        : "manual",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs).argmax(1).numpy())
        labels.extend(lbls.numpy())
    y_pred = np.array(preds)
    y_true = np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# CPU Inference Timing
# time.perf_counter() is correct for CPU — CUDA events are not applicable.
# Separate warmup phase stabilises OS scheduler, branch predictor, and caches.
# ══════════════════════════════════════════════════════════════════════════════
def measure_cpu_throughput(model: nn.Module, n: int = INFERENCE_RUNS) -> dict:
    model = copy.deepcopy(model).cpu().eval()
    dummy_batch  = torch.randn(BATCH_SIZE, 3, 32, 32)
    dummy_single = torch.randn(1,          3, 32, 32)

    with torch.no_grad():
        for _ in range(WARMUP_RUNS):
            model(dummy_batch)
            model(dummy_single)

    batch_timings, single_timings = [], []
    with torch.no_grad():
        for _ in range(n):
            t0 = time.perf_counter()
            model(dummy_batch)
            batch_timings.append((time.perf_counter() - t0) * 1000)

        for _ in range(n):
            t0 = time.perf_counter()
            model(dummy_single)
            single_timings.append((time.perf_counter() - t0) * 1000)

    batch_ms  = float(np.mean(batch_timings))
    single_ms = float(np.mean(single_timings))
    return {
        "single_img_cpu_ms"  : round(single_ms, 4),
        "batch128_cpu_ms"    : round(batch_ms, 4),
        "per_img_cpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : "wall-clock (time.perf_counter) — CPU only; CUDA events not applicable",
        "warmup_runs"        : WARMUP_RUNS,
        "timing_runs"        : n,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Canonical Result Entry Builder
# Mirrors Script 2 schema exactly so downstream comparison is straightforward.
# ══════════════════════════════════════════════════════════════════════════════
def build_entry(metrics: dict, params: dict,
                size_mb: float, flops_G: float,
                inference_ms: dict) -> dict:
    return {
        "accuracy"      : round(metrics["accuracy"],  6),
        "precision"     : round(metrics["precision"], 6),
        "recall"        : round(metrics["recall"],    6),
        "f1"            : round(metrics["f1"],        6),
        "params"        : int(params["params_total"]),
        "params_nonzero": int(params["params_nonzero"]),
        "size_mb"       : round(size_mb, 6),
        "flops_G"       : round(flops_G, 9),
        "inference_ms"  : inference_ms,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Calibration
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def calibrate(model: nn.Module, loader: DataLoader,
              max_batches: int = CALIB_BATCHES) -> None:
    """
    Run forward passes with calibration data to collect activation statistics.
    The prepared model accumulates min/max (or histogram) per observer.
    No gradients needed — observers update their running stats in-place.
    """
    model.eval()
    for i, (inputs, _) in enumerate(loader):
        model(inputs)
        if i + 1 >= max_batches:
            break


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    SEP = "=" * 65

    print(f"\n{SEP}")
    print("  PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10")
    print("  Observer : MinMax (per-tensor activation, per-channel weight)")
    print("  Backend  : onednn (CPU INT8 SIMD)")
    print(f"  Calib    : {CALIB_SIZE} images ({CALIB_BATCHES} batches)")
    print(f"  Timing   : {INFERENCE_RUNS} runs, {WARMUP_RUNS} warmup")
    print(SEP)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs — computed once on FP32; reused for INT8 (identical op count) ───
    print("\n  Computing FLOPs (identical for FP32 and INT8) ...")
    flops_info = count_flops(fp32_model)
    flops_G    = flops_info["flops_G"]
    print(f"  FLOPs : {flops_G:.4f} GFLOPs  "
          f"(tool: {flops_info['tool']})")

    # ── [1/2] FP32 Baseline ───────────────────────────────────────────────────
    print("\n  [1/2] FP32 Baseline")
    base_metrics = evaluate(fp32_model, test_loader)
    base_infer   = measure_cpu_throughput(fp32_model)
    base_disk    = disk_size_mb(fp32_model)
    base_params  = count_params(fp32_model, model_label="fp32_baseline")

    print(
        f"        Acc={base_metrics['accuracy']:.4f}  "
        f"Disk={base_disk:.2f} MB  "
        f"Batch={base_infer['batch128_cpu_ms']:.1f} ms  "
        f"GFLOPs={flops_G:.3f}"
    )

    # ── [2/2] Static PTQ — MinMax ─────────────────────────────────────────────
    print("\n  [2/2] Static PTQ — MinMax observer")
    print("        Inserting observers → calibrating → converting to INT8 ...")

    example  = torch.randn(1, 3, 32, 32)
    model    = copy.deepcopy(fp32_model).cpu().eval()

    # prepare_fx inserts MinMaxObserver nodes at every quantizable op boundary.
    # example_inputs must be a tuple-of-args for correct FX tracing.
    prepared = prepare_fx(model, {"": BEST_QCONFIG}, (example,))

    # Calibration: forward passes collect per-tensor/per-channel min/max.
    # A fresh calib_loader is used so the iterator is never exhausted
    # between multiple methods (critical for fair multi-method comparisons).
    calibrate(prepared, calib_loader)

    # convert_fx: freeze S and Z, replace FakeQuantize/observer nodes
    # with genuine INT8 quantized ops backed by onednn SIMD kernels.
    q_model = convert_fx(prepared)
    q_model.eval()

    print("        Evaluating on test set ...")
    q_metrics = evaluate(q_model, test_loader)
    q_infer   = measure_cpu_throughput(q_model)
    q_disk    = disk_size_mb(q_model)
    q_params  = count_params(q_model, model_label="static_minmax")

    # FLOPs: thop cannot hook quantized FX ops → reuse FP32 value (correct).
    # Quantization does not change the computation graph or op count.
    q_flops_info = count_flops(q_model)
    q_flops_G    = q_flops_info["flops_G"] if q_flops_info["flops_G"] > 0.01 else flops_G

    # ── Derived comparison metrics ────────────────────────────────────────────
    acc_drop      = base_metrics["accuracy"] - q_metrics["accuracy"]
    batch_speedup = base_infer["batch128_cpu_ms"] / q_infer["batch128_cpu_ms"]
    size_ratio    = base_disk / q_disk

    # ── Build entries using canonical schema ──────────────────────────────────
    fp32_entry = build_entry(
        metrics      = base_metrics,
        params       = base_params,
        size_mb      = base_disk,
        flops_G      = flops_G,
        inference_ms = base_infer,
    )

    q_entry = build_entry(
        metrics      = q_metrics,
        params       = q_params,
        size_mb      = q_disk,
        flops_G      = q_flops_G,
        inference_ms = q_infer,
    )
    # Attach INT8-specific note inside the entry
    q_entry["notes"] = (
        "single_img_cpu_ms higher than FP32 is expected at batch_size=1 — "
        "FX op-dispatch overhead dominates small batches. "
        "Batch throughput is the meaningful deployment metric."
    )

    # ── Best result section ───────────────────────────────────────────────────
    # Since this script runs a single method (static_minmax), the best result
    # IS static_minmax. The section captures the key comparison figures in one
    # place for the report.
    best_result = {
        "method"          : "static_minmax",
        "observer"        : "MinMaxObserver (per-tensor activation, per-channel weight)",
        "backend"         : "onednn (CPU INT8 SIMD)",
        "accuracy"        : q_entry["accuracy"],
        "f1"              : q_entry["f1"],
        "acc_drop"        : round(acc_drop, 6),
        "size_mb"         : q_entry["size_mb"],
        "size_reduction_x": round(size_ratio, 4),
        "batch128_cpu_ms" : q_entry["inference_ms"]["batch128_cpu_ms"],
        "throughput_imgs_sec": q_entry["inference_ms"]["throughput_imgs_sec"],
        "batch_speedup_x" : round(batch_speedup, 4),
        "flops_G"         : q_entry["flops_G"],
        "params"          : q_entry["params"],
        "params_nonzero"  : q_entry["params_nonzero"],
        "selection_reason": (
            "Only method evaluated in this script. "
            "Chosen over dynamic PTQ (sweep result): static PTQ quantizes "
            "Conv2d + Linear + residual adds in INT8; dynamic PTQ only "
            "quantizes Linear weights, leaving Conv2d (dominant compute) in FP32, "
            "giving negligible size/speed benefit."
        ),
    }

    # ── Full output ───────────────────────────────────────────────────────────
    output = {
        "fp32_baseline" : fp32_entry,
        "static_minmax" : q_entry,
        "_best_result"  : best_result,
        "_meta": {
            "script"          : "PTQ CPU best config — static_minmax",
            "calib_images"    : CALIB_SIZE,
            "calib_batches"   : CALIB_BATCHES,
            "inference_runs"  : INFERENCE_RUNS,
            "warmup_runs"     : WARMUP_RUNS,
            "flops_tool"      : flops_info["tool"],
            "flops_note"      : (
                "FLOPs identical for FP32 and INT8 — quantization changes "
                "storage dtype, not arithmetic op count. "
                "INT8 batch speedup comes from VNNI/SIMD packing on CPU "
                "(4 INT8 MACs per instruction vs 1 FP32 FMA)."
            ),
            "timing_note"     : (
                "CPU-only script. time.perf_counter() is appropriate here. "
                "CUDA events are not applicable."
            ),
            "size_note"       : (
                "Both models serialized via torch.save(model) — full model "
                "including quantization config — for a fair apples-to-apples "
                "size comparison."
            ),
            "param_count_note": (
                "FX-quantized model: .weight() callable params counted via "
                "module-path key (not data_ptr) to avoid aliasing bugs. "
                "Standard params deduplicated by id(p)."
            ),
        },
    }

    # ── Save JSON ─────────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  JSON  → {OUTPUT_JSON}")

    # ── Save model ────────────────────────────────────────────────────────────
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    model_path = os.path.join(MODEL_SAVE_DIR, "resnet50_static_minmax.pt")
    torch.save(q_model, model_path)
    saved_mb = os.path.getsize(model_path) / 1024 ** 2
    print(f"  Model → {model_path}  ({saved_mb:.2f} MB)")

    # ── Console summary ───────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("  RESULTS SUMMARY")
    print(SEP)
    print(f"  {'Metric':<28} {'FP32':>12}  {'static_minmax':>14}")
    print(f"  {'-'*58}")
    print(f"  {'Accuracy':<28} {base_metrics['accuracy']:>12.4f}  {q_metrics['accuracy']:>14.4f}")
    print(f"  {'F1 (macro)':<28} {base_metrics['f1']:>12.4f}  {q_metrics['f1']:>14.4f}")
    print(f"  {'Params (total)':<28} {base_params['params_total']:>12,}  {q_params['params_total']:>14,}")
    print(f"  {'Params (nonzero)':<28} {base_params['params_nonzero']:>12,}  {q_params['params_nonzero']:>14,}")
    print(f"  {'Disk size (MB)':<28} {base_disk:>12.2f}  {q_disk:>14.2f}")
    print(f"  {'Batch latency (ms)':<28} {base_infer['batch128_cpu_ms']:>12.1f}  {q_infer['batch128_cpu_ms']:>14.1f}")
    print(f"  {'Throughput (imgs/s)':<28} {base_infer['throughput_imgs_sec']:>12.1f}  {q_infer['throughput_imgs_sec']:>14.1f}")
    print(f"  {'Single-img latency (ms)':<28} {base_infer['single_img_cpu_ms']:>12.4f}  {q_infer['single_img_cpu_ms']:>14.4f}")
    print(f"  {'GFLOPs':<28} {flops_G:>12.4f}  {q_flops_G:>14.4f}")
    print(f"  {'-'*58}")
    print(f"  Accuracy drop    : {acc_drop:+.6f}")
    print(f"  Batch speedup    : {batch_speedup:.4f}×")
    print(f"  Size reduction   : {size_ratio:.4f}×")
    print(SEP + "\n")

    return output, q_model


# %%
if __name__ == "__main__":
    report, best_model = main()


  PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10
  Observer : MinMax (per-tensor activation, per-channel weight)
  Backend  : onednn (CPU INT8 SIMD)
  Calib    : 1024 images (8 batches)
  Timing   : 200 runs, 20 warmup

  Computing FLOPs (identical for FP32 and INT8) ...
  FLOPs : 2.6232 GFLOPs  (tool: thop)

  [1/2] FP32 Baseline
      count_params [fp32_baseline]: total=23,520,842  nonzero=23,520,842  sparsity=0.0000
        Acc=0.9320  Disk=90.05 MB  Batch=781.0 ms  GFLOPs=2.623

  [2/2] Static PTQ — MinMax observer
        Inserting observers → calibrating → converting to INT8 ...


W0618 16:27:06.649000 63932 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


        Evaluating on test set ...
      count_params [static_minmax]: total=23,467,712  nonzero=20,599,612  sparsity=0.1222

  JSON  → __1__PTQ_CPU.json
  Model → __1__saved_models_PTQ_CPU\resnet50_static_minmax.pt  (22.99 MB)

  RESULTS SUMMARY
  Metric                               FP32   static_minmax
  ----------------------------------------------------------
  Accuracy                           0.9320          0.9329
  F1 (macro)                         0.9319          0.9328
  Params (total)                 23,520,842      23,467,712
  Params (nonzero)               23,520,842      20,599,612
  Disk size (MB)                      90.05           22.99
  Batch latency (ms)                  781.0           241.6
  Throughput (imgs/s)                 163.9           529.9
  Single-img latency (ms)           14.1412         92.2007
  GFLOPs                             2.6232          2.6232
  ----------------------------------------------------------
  Accuracy drop    : -0.000900
